# Neural Network Models for Demand Forecasting

Two neural network architectures evaluated on the same train/val/test splits as the tree-based models:

| Model | Architecture | Input |
|-------|-------------|-------|
| **MLP** (Deep Fully-Connected) | 4 hidden layers, BatchNorm, Dropout | Same 97 tabular features as LightGBM |
| **LSTM** | 2-layer LSTM + dense head | 8-week sequences of 12 time-varying features (no pre-computed lags — LSTM learns temporal dependencies itself) |

Both are compared against the best tree model (RandomForest, Test RMSE = 6.2333).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings, os, time
warnings.filterwarnings('ignore')

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    print(f'PyTorch {torch.__version__}')
except ImportError:
    raise ImportError("Install PyTorch: pip install torch")

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import joblib

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 130
os.makedirs('plots/neural', exist_ok=True)

BASE  = '/Users/kamilaya/Desktop/thesis'
SEED  = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def rmse(a, b):
    return np.sqrt(mean_squared_error(np.asarray(a), np.asarray(b)))

def nz_mape(a, b):
    a, b = np.asarray(a), np.asarray(b)
    mask = a > 0
    return np.mean(np.abs((a[mask] - b[mask]) / a[mask])) * 100 if mask.sum() else np.nan

def score(label, y_true, y_pred, split):
    y_pred = np.clip(y_pred, 0, None)
    return {'model': label, 'split': split,
            'RMSE': round(rmse(y_true, y_pred), 4),
            'MAE':  round(mean_absolute_error(y_true, y_pred), 4),
            'NZ-MAPE (%)': round(nz_mape(y_true, y_pred), 2)}


PyTorch 2.11.0
Device: cpu


---
## Part 1: Deep MLP (Fully-Connected Network)

Uses the **same 97 tabular features** as LightGBM — a direct architecture comparison.

Architecture:
```
Input(97) → [Linear(512) → BN → ReLU → Drop(0.3)]
          → [Linear(256) → BN → ReLU → Drop(0.3)]
          → [Linear(128) → BN → ReLU → Drop(0.2)]
          → [Linear(64)  → ReLU]
          → Linear(1)
```
Training: AdamW, lr=1e-3, ReduceLROnPlateau, early stopping (patience=15).


In [2]:
# Load same scaled splits used by models.ipynb
train = pd.read_csv(f'{BASE}/tshirts_train.csv', parse_dates=['week_start'])
val   = pd.read_csv(f'{BASE}/tshirts_val.csv',   parse_dates=['week_start'])
test  = pd.read_csv(f'{BASE}/tshirts_test.csv',  parse_dates=['week_start'])
TARGET = 'log_sales_volume'

OHE_COLS = [c for c in train.columns if any(c.startswith(p) for p in [
    'index_group_name_', 'colour_group_name_',
    'graphical_appearance_name_', 'perceived_colour_value_name_'])]

BASE_FEATURES = [
    'avg_weekly_price', 'real_price', 'price_vs_median',
    'week_of_year', 'month', 'quarter', 'is_spring_summer', 'is_sale_season', 'covid',
    'product_age_weeks', 'sales_lag1', 'sales_lag2', 'sales_lag4',
    'sales_rolling4_mean', 'sales_rolling4_std',
] + OHE_COLS

features = [f for f in BASE_FEATURES if f in train.columns]
print(f'MLP features: {len(features)}')

X_tr = train[features].values.astype(np.float32);  y_tr = train[TARGET].values.astype(np.float32)
X_va = val[features].values.astype(np.float32);    y_va = val[TARGET].values.astype(np.float32)
X_te = test[features].values.astype(np.float32);   y_te = test[TARGET].values.astype(np.float32)

y_val_raw  = np.expm1(val[TARGET].values)
y_test_raw = np.expm1(test[TARGET].values)


KeyboardInterrupt: 

In [ ]:
class DeepMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512, 256),       nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),        nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp(model, X_tr, y_tr, X_va, y_va_raw,
              max_epochs=120, batch_size=4096, patience=15):
    opt  = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    crit = nn.MSELoss()

    ds  = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
    ldr = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0)

    X_va_t = torch.tensor(X_va).to(device)

    best_rmse, best_state, no_imp = float('inf'), None, 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in ldr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()

        model.eval()
        with torch.no_grad():
            vp = np.expm1(np.clip(model(X_va_t).cpu().numpy(), 0, None))
        v_rmse = rmse(y_va_raw, vp)
        history.append(v_rmse)
        sched.step(v_rmse)

        if v_rmse < best_rmse:
            best_rmse = v_rmse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1

        if epoch % 20 == 0 or no_imp == 0:
            print(f'  Epoch {epoch:3d} | Val RMSE: {v_rmse:.4f}  (best: {best_rmse:.4f})')

        if no_imp >= patience:
            print(f'  Early stop at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    return history


In [ ]:
print('Training Deep MLP...')
t0 = time.time()
mlp = DeepMLP(len(features)).to(device)
mlp_history = train_mlp(mlp, X_tr, y_tr, X_va, y_val_raw)
print(f'Done in {time.time()-t0:.0f}s')

mlp.eval()
with torch.no_grad():
    mlp_val_pred  = np.expm1(np.clip(mlp(torch.tensor(X_va).to(device)).cpu().numpy(), 0, None))
    mlp_test_pred = np.expm1(np.clip(mlp(torch.tensor(X_te).to(device)).cpu().numpy(), 0, None))

print()
print('MLP Results:')
for split, y_true, y_pred in [('val', y_val_raw, mlp_val_pred), ('test', y_test_raw, mlp_test_pred)]:
    print(f'  {split:4s}  RMSE={rmse(y_true,y_pred):.4f}  MAE={mean_absolute_error(y_true,y_pred):.4f}'
          f'  NZ-MAPE={nz_mape(y_true,y_pred):.1f}%')

torch.save(mlp.state_dict(), f'{BASE}/saved_models/MLP_baseline.pt')
print('Model saved.')


Training Deep MLP...
  Epoch   1 | Val RMSE: 7.9368  (best: 7.9368)
  Epoch   3 | Val RMSE: 7.8918  (best: 7.8918)
  Epoch   5 | Val RMSE: 6.8062  (best: 6.8062)
  Epoch   7 | Val RMSE: 6.0354  (best: 6.0354)
  Epoch  12 | Val RMSE: 5.6267  (best: 5.6267)
  Epoch  20 | Val RMSE: 7.7267  (best: 5.6267)
  Early stop at epoch 27
Done in 46s

MLP Results:
  val   RMSE=5.6267  MAE=1.1101  NZ-MAPE=65.7%
  test  RMSE=6.7040  MAE=1.2143  NZ-MAPE=69.6%
Model saved.


---
## Part 2: LSTM (Sequence Model)

Unlike the MLP, the LSTM receives **raw time-varying features** (no pre-computed lag features) 
and learns temporal dependencies directly from the sequence of past weeks.

- **Sequence length:** 8 weeks of history → predict week 9
- **Features per timestep (12):** price, real_price, price_vs_median, week_of_year, month,
  is_spring_summer, is_sale_season, covid, product_age_weeks, HICP, unemployment, CCI,
  plus `log_sales_volume` of that past week (the series itself)
- **Architecture:** LSTM(128, 2 layers, dropout=0.2) → Linear(64) → ReLU → Linear(1)
- Articles with fewer than 9 weeks of history are zero-padded (cold-start case)


In [ ]:
# Load unscaled full grid — LSTM needs raw series to build sequences
df = pd.read_csv(f'{BASE}/tshirts_preprocessed.csv', parse_dates=['week_start'])
df = df.sort_values(['article_id', 'week_start']).reset_index(drop=True)

TRAIN_END = pd.Timestamp('2019-04-30')
VAL_END   = pd.Timestamp('2019-12-31')
SEQ_LEN   = 8

# Time-varying features (NO pre-computed lags — LSTM learns these itself)
LSTM_FEATS = [
    'avg_weekly_price', 'real_price', 'price_vs_median',
    'week_of_year', 'month', 'is_spring_summer', 'is_sale_season', 'covid',
    'product_age_weeks',
    'eurozone_hicp', 'eurozone_unemployment_rate', 'eurozone_cci',
    'log_sales_volume',   # past sales as input feature (the time series signal)
]
LSTM_FEATS = [f for f in LSTM_FEATS if f in df.columns]
N_FEATS = len(LSTM_FEATS)
print(f'LSTM features per timestep: {N_FEATS}')

# Scale on training data only (prevent leakage)
scaler = StandardScaler()
train_mask = df['week_start'] <= TRAIN_END
scaler.fit(df.loc[train_mask, LSTM_FEATS])
df_scaled = df.copy()
df_scaled[LSTM_FEATS] = scaler.transform(df[LSTM_FEATS])


LSTM features per timestep: 13


In [ ]:
# Build per-article sequences
# For each (article, week_i): input = features[week i-SEQ_LEN : week i], target = log_sales[week i]
# Zero-pad articles with < SEQ_LEN prior weeks (cold-start)

X_tr_seq, y_tr_seq = [], []
X_va_seq, y_va_seq = [], []
X_te_seq, y_te_seq = [], []

for art_id, grp in df_scaled.groupby('article_id'):
    grp = grp.sort_values('week_start').reset_index(drop=True)
    feats   = grp[LSTM_FEATS].values.astype(np.float32)       # (n_weeks, N_FEATS)
    targets = grp['log_sales_volume'].values.astype(np.float32)
    weeks   = grp['week_start'].values

    for i in range(len(grp)):
        week_ts = pd.Timestamp(weeks[i])
        if i < SEQ_LEN:
            pad = np.zeros((SEQ_LEN - i, N_FEATS), dtype=np.float32)
            seq = np.vstack([pad, feats[:i]])
        else:
            seq = feats[i - SEQ_LEN: i]     # (SEQ_LEN, N_FEATS)

        y = targets[i]
        if week_ts <= TRAIN_END:
            X_tr_seq.append(seq); y_tr_seq.append(y)
        elif week_ts <= VAL_END:
            X_va_seq.append(seq); y_va_seq.append(y)
        else:
            X_te_seq.append(seq); y_te_seq.append(y)

X_tr_seq = np.array(X_tr_seq); y_tr_seq = np.array(y_tr_seq, dtype=np.float32)
X_va_seq = np.array(X_va_seq); y_va_seq = np.array(y_va_seq, dtype=np.float32)
X_te_seq = np.array(X_te_seq); y_te_seq = np.array(y_te_seq, dtype=np.float32)

y_va_seq_raw = np.expm1(y_va_seq)
y_te_seq_raw = np.expm1(y_te_seq)

print(f'Train sequences: {len(X_tr_seq):,}')
print(f'Val  sequences: {len(X_va_seq):,}')
print(f'Test sequences: {len(X_te_seq):,}')
print(f'Sequence shape: {X_tr_seq[0].shape}  (SEQ_LEN x N_FEATS)')


Train sequences: 259,809
Val  sequences: 275,555
Test sequences: 299,174
Sequence shape: (8, 13)  (SEQ_LEN x N_FEATS)


In [ ]:
class SalesLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.ReLU(), nn.Linear(64, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)          # (batch, seq_len, hidden)
        last   = out[:, -1, :]         # last timestep
        return self.head(last).squeeze(-1)


def train_lstm(model, X_tr, y_tr, X_va, y_va_raw,
               max_epochs=80, batch_size=512, patience=12):
    opt  = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=4, factor=0.5)
    crit = nn.MSELoss()

    ds  = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
    ldr = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0)
    X_va_t = torch.tensor(X_va).to(device)

    best_rmse, best_state, no_imp = float('inf'), None, 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in ldr:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()

        model.eval()
        with torch.no_grad():
            vp = np.expm1(np.clip(model(X_va_t).cpu().numpy(), 0, None))
        v_rmse = rmse(y_va_raw, vp)
        history.append(v_rmse)
        sched.step(v_rmse)

        if v_rmse < best_rmse:
            best_rmse = v_rmse; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}; no_imp = 0
        else:
            no_imp += 1

        if epoch % 10 == 0 or no_imp == 0:
            print(f'  Epoch {epoch:3d} | Val RMSE: {v_rmse:.4f}  (best: {best_rmse:.4f})')

        if no_imp >= patience:
            print(f'  Early stop at epoch {epoch}'); break

    model.load_state_dict(best_state)
    return history


In [ ]:
print('Training LSTM...')
t0 = time.time()
lstm_model = SalesLSTM(input_size=N_FEATS).to(device)
lstm_history = train_lstm(lstm_model, X_tr_seq, y_tr_seq, X_va_seq, y_va_seq_raw)
print(f'Done in {time.time()-t0:.0f}s')

lstm_model.eval()
with torch.no_grad():
    lstm_val_pred  = np.expm1(np.clip(lstm_model(torch.tensor(X_va_seq).to(device)).cpu().numpy(), 0, None))
    lstm_test_pred = np.expm1(np.clip(lstm_model(torch.tensor(X_te_seq).to(device)).cpu().numpy(), 0, None))

print()
print('LSTM Results:')
for split, y_true, y_pred in [('val', y_va_seq_raw, lstm_val_pred), ('test', y_te_seq_raw, lstm_test_pred)]:
    print(f'  {split:4s}  RMSE={rmse(y_true,y_pred):.4f}  MAE={mean_absolute_error(y_true,y_pred):.4f}'
          f'  NZ-MAPE={nz_mape(y_true,y_pred):.1f}%')

torch.save(lstm_model.state_dict(), f'{BASE}/saved_models/LSTM_baseline.pt')
print('Model saved.')


Training LSTM...
  Epoch   1 | Val RMSE: 8.1766  (best: 8.1766)
  Epoch   2 | Val RMSE: 6.8938  (best: 6.8938)
  Epoch   3 | Val RMSE: 6.7443  (best: 6.7443)
  Epoch   4 | Val RMSE: 6.5946  (best: 6.5946)
  Epoch  10 | Val RMSE: 6.9158  (best: 6.5946)
  Early stop at epoch 16
Done in 209s

LSTM Results:
  val   RMSE=6.5946  MAE=1.4307  NZ-MAPE=92.0%
  test  RMSE=14.0353  MAE=2.5402  NZ-MAPE=89.0%
Model saved.


In [ ]:
# ── Full comparison table ─────────────────────────────────────────────────────
rows = []

# New NN models
for split, y_true_mlp, y_pred_mlp, y_true_lstm, y_pred_lstm in [
    ('val',  y_val_raw,  mlp_val_pred,  y_va_seq_raw, lstm_val_pred),
    ('test', y_test_raw, mlp_test_pred, y_te_seq_raw, lstm_test_pred),
]:
    rows.append(score('MLP (deep FC)',  y_true_mlp,  y_pred_mlp,  split))
    rows.append(score('LSTM (seq=8)',   y_true_lstm, y_pred_lstm, split))

# Reference tree models (from saved results)
ref = pd.read_csv(f'{BASE}/model_results.csv')
for _, r in ref[ref['feature_set'] == 'Baseline (no macro)'].iterrows():
    rows.append({'model': r['model'] + ' (tree)', 'split': r['split'],
                 'RMSE': r['RMSE'], 'MAE': r['MAE'], 'NZ-MAPE (%)': r['NZ-MAPE (%)']})

cmp = pd.DataFrame(rows)
for split in ['val', 'test']:
    sub = cmp[cmp['split'] == split].sort_values('RMSE').reset_index(drop=True)
    sub.index += 1
    print(f'\n{"="*60}')
    print(f'  {split.upper()} SET')
    print(f'{"="*60}')
    print(sub[['model', 'RMSE', 'MAE', 'NZ-MAPE (%)']].to_string())



  VAL SET
                 model    RMSE     MAE  NZ-MAPE (%)
1      LightGBM (tree)  5.2051  0.9669    59.690000
2  RandomForest (tree)  5.2160  0.9941    67.160000
3       XGBoost (tree)  5.2765  0.9757    59.080000
4        MLP (deep FC)  5.6267  1.1101    65.720000
5         LSTM (seq=8)  6.5946  1.4307    92.019997

  TEST SET
                 model     RMSE     MAE  NZ-MAPE (%)
1  RandomForest (tree)   6.2333  1.0911    75.280000
2      LightGBM (tree)   6.2436  1.0702    68.550000
3       XGBoost (tree)   6.3489  1.0879    67.910000
4        MLP (deep FC)   6.7040  1.2143    69.570000
5         LSTM (seq=8)  14.0353  2.5402    88.959999


In [ ]:
# ── Plot 1: Training curves ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, hist, label, color in [
    (axes[0], mlp_history,  'MLP',  'steelblue'),
    (axes[1], lstm_history, 'LSTM', 'mediumseagreen'),
]:
    ax.plot(hist, color=color, linewidth=2)
    ax.axhline(min(hist), color='tomato', linestyle='--', linewidth=1.2,
               label=f'Best: {min(hist):.4f}')
    ax.set_title(f'{label} — Validation RMSE per Epoch', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Val RMSE (original scale)')
    ax.legend()

plt.tight_layout()
plt.savefig('plots/neural/N1_training_curves.png')
plt.close()

# ── Plot 2: RMSE comparison bar chart ─────────────────────────────────────────
test_cmp = cmp[cmp['split'] == 'test'].sort_values('RMSE', ascending=False).reset_index(drop=True)
colors = ['mediumseagreen' if 'MLP' in m or 'LSTM' in m else '#3498db' for m in test_cmp['model']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(test_cmp['model'], test_cmp['RMSE'], color=colors, edgecolor='white', alpha=0.88)
for bar, v in zip(bars, test_cmp['RMSE']):
    ax.text(v + 0.01, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=9.5, fontweight='bold')
ax.set_xlabel('Test RMSE (original scale)')
ax.set_title('Neural Networks vs. Tree Models — Test RMSE\n(green = NN, blue = tree)', fontweight='bold')
plt.tight_layout()
plt.savefig('plots/neural/N2_rmse_comparison.png')
plt.close()
print('Plots saved to plots/neural/')


Plots saved to plots/neural/
